# Reliability Calculator for Distributed System Missions

This code is the one to be run. This is the file where the inputs are set and the iterations run
based on the user's desire. The inputs define the parameters of the DSM and simulation, as well as
outputing the reliability graphs. This file uses the functions set on Reliability.py. -- add more explanations--

#### Setup
The program begins by setting the necessary libraries and tools in order for it to work. These variables mentioned start blank at the beginning of each iteration. 
- iteration: string that asks whether the user wants to try another DSM configuration
- graph_labels: array that compiles the configuration parameters entered to be lated used as the graph's label.
- cont: variable used in the iteration of a final graph.
- year_to_month: a conversion, self explained.


In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import scipy as sp
from IPython.display import Markdown
from IPython.display import Latex
from IPython.display import Image, SVG
import pandas as pd
R_simulation = []
iteration = ''
graph_labels = []
year_to_month = 12

## Inputs

### Mission Time
This input sets how much time does the distributed mission lasts, in years. This provides the timeframe for the simulation to work.

In [ ]:
Mission_time = 8
Time_months = np.arange(0, Mission_time * year_to_month)

First batch of satellites launched

In [ ]:
DSM_initial_amount = [7, 5, 2 , 2, 2]

### CubeSat Reliability Curve

The methodology applied in this work is based of Bouwmeester et al's work, starting by defining the relationship between subsystems in a CubeSat as in series, where:
$$
R_{sys} = \prod_{n}^{i}R_{i}
$$
Where $R_{sys}$ represents the reliability of the CubeSat and $R_i$ represents the reliability of the n-th subsystem, both obtained by Bouwmeester et al. At the same time the model of the reliability obtained by the authors corresponds to a Lognormal-Gompertz product mixture, showned up next:
$$
R_{subsys} = e^{-\eta\left ( e^{\frac{t}{\theta}} -1 \right ) }\cdot \frac{1}{\sigma\sqrt{2\pi}}\int_{0}^{t}\frac{2}{x}e^{-\frac{\left ( \ln{x} - \mu \right )^2}{4\sigma^2}}dx
$$
Where $\theta$, $\eta$, $\sigma$, $\mu$ are variables that determines the curve for each subsystems and determined by the authors. In the code down below they are expressed as arrays where each column is for each subsystem in this order:
- Attitude, Determination and Control Subsystem (ADCS)
- Control and Data Handling Subsystem (CDHS)
- Communications Subsystem (COMMS)
- Structure and Deployment Subsystem (STS&DepS)
- Payload Subsystem (P/L)
- Electric Power Subsystem (EPS)

In [ ]:
mu = [15.4, 11.5, 13.7, 14.3, 14.3, 9.4]    #Log-normal \mu
sigma = [10, 8.39, 9.79, 9.21, 9.21, 8.18]  #Log-normal \sigma_1
theta = [2.6, 8.1, 2.6, 2.7, 2.7, 2.9]      #Gomperts \theta
nu = [9.1e-5, 0.0167, 8.3e-5, 0.00011, 0.00011, 0.00011]    #Gompertz \nu

They are later compiled into this function that provides the reliability of the CubeSat from its subsystems, given by Bouwmeester et al in 2022 and taking into conideration the use of redundancy on its most.

In [ ]:
def Reliability_CubeSat(EPS_redundancy, Time_months): # Subsystem level realibility inputs Source: Bouwmeester et al 2022 - Fig 11
    R_CubeSat = 1
#Subsystem redundancy true is redundant (red) and false (off) is no subsystem redundancy
    #Only analisis case for EPS redundancy, due to its lower reliability over time
    counter: int # counter is a variable used for the loop as counter
    for counter in np.arange(len(mu)):
        if EPS_redundancy and counter == 5:  #Counter ==5 is the position in the input vector for EPS reliability
            R_CubeSat *= 1 - np.square(
                1 - sp.stats.lognorm.sf(Time_months, sigma[counter], scale=np.exp(mu[counter])) * sp.stats.gompertz.sf(Time_months, nu[
                    counter] / theta[counter], scale=theta[counter]))
        else: #No redundancy
            R_CubeSat *= sp.stats.lognorm.sf(Time_months, sigma[counter], scale=np.exp(mu[counter])) * sp.stats.gompertz.sf(Time_months, nu[
                counter] / theta[counter], scale=theta[counter])
    return R_CubeSat #outputs the system level reliability curve

## Process

### Mission Strategy

#### Subsystem Redundancy
As mentioned in the previous chapter, several authors establish as a primary strategy to improve
the reliability of CubeSat elements the inclusion of redundancies in the EPS subsystem. This in
practice translates into a standby system that, for simplification purposes, is a parallel system. This element is expressed by the boolean variable **EPS_redundancy** that defines the existence or absence in redundancy in EPS subsystem.

In [ ]:
EPS_redundancy = [False, False, False, True, False]

#### Phased Deployment
The periodic launch strategy consists of the constant maintenance of the mission by launching more elements into space in a period T when integrated into the system. Examples in practice are those discussed in Chapter 1 that practice this strategy to maintain their services for a long time. This, in practice, generates a constant replacement of new elements entering the space segment, compensating for the possibility of failure over time.

- Phase_Deployment: Boolean variable, defines the use of phased deployment as mission strategy.
- relaunch_rate: Amount of months between launches of new batches to the DSM
- DSM_relaunch_amount: Amount of satellites to be thrown per relaunch. 

In [ ]:
Phase_Deployment = [True, True, True, True, False]
relaunch_rate = [24, 12, 30, 24, 0]
DSM_relaunch_amount = [1, 2, 1, 1, 0]

---- Explain here the code written --- Remove the prints and use display markdown ---

In [ ]:
simulation_results = []
for iteration in np.arange(len(DSM_initial_amount)):
    R_DSM_6m = 0
    R_DSM_12m = 0
    R_DSM_18m = 0
    R_DSM_24m = 0
    if Phase_Deployment[iteration] == True:
        R_DSM = []
        for Current_Time in Time_months:
            R_elem = np.ones(DSM_initial_amount[iteration]) * Reliability_CubeSat(EPS_redundancy[iteration], Current_Time / year_to_month)
            Deployment_Dates = np.arange(relaunch_rate[iteration], (Mission_time * year_to_month) + 1, relaunch_rate[iteration], dtype=int) # Array that gives the dates for phased deployment, in months

            for a in Deployment_Dates: # if the current time is over the deployment date, it adds new elements to the DSM, else continues
                if int(Current_Time) >= int(a):
                    R_elem = np.append(R_elem, np.ones(DSM_relaunch_amount[iteration]) * Reliability_CubeSat(EPS_redundancy[iteration], Current_Time / year_to_month - float(a / year_to_month))).tolist()
                else:
                    continue #sets DSM size and cubesats reliabilities, updating it to the current time
            
            R_DSM_partial = 1
            for R_CubeSat in np.arange(len(R_elem)):
                R_DSM_partial *= 1 - R_elem[R_CubeSat]
            R_DSM.append(1 - R_DSM_partial)
            if Current_Time == 6:
                R_DSM_6m = np.round(1 - R_DSM_partial, 3)
            elif Current_Time == 12:
                R_DSM_12m = np.round(1 - R_DSM_partial, 3)
            elif Current_Time == 18:
                R_DSM_18m = np.round(1 - R_DSM_partial, 3)
            elif Current_Time == 24:
                R_DSM_24m = np.round(1 - R_DSM_partial, 3) #gets DSM reliability and appends on the array
    else:
        R_DSM = []
        for Current_Time in Time_months:
            R_elem = np.ones(DSM_initial_amount[iteration]) * Reliability_CubeSat(EPS_redundancy[iteration], Current_Time  / year_to_month) #sets DSM size and cubesats reliabilities
            
            R_DSM_partial = 1
            for R_CubeSat in np.arange(len(R_elem)):
                R_DSM_partial *= 1 - R_elem[R_CubeSat]
            R_DSM.append(1 - R_DSM_partial) #gets DSM reliability and appends on the array
            relaunch_rate[iteration] = 0
            DSM_relaunch_amount[iteration] = 0
            if Current_Time == 3:
                R_DSM_6m = np.round(1 - R_DSM_partial, 3)
            elif Current_Time == 6:
                R_DSM_12m = np.round(1 - R_DSM_partial, 3)
            elif Current_Time == 12:
                R_DSM_18m = np.round(1 - R_DSM_partial, 3)
            elif Current_Time == 18:
                R_DSM_24m = np.round(1 - R_DSM_partial, 3)
    
    result = {'Scenario': iteration + 1, 'Initial Batch': DSM_initial_amount[iteration], 'Subsystem Redundancy': str(EPS_redundancy[iteration]), 'Relaunch Rate': relaunch_rate[iteration], 'Reliability at 6 months': R_DSM_6m, 'Reliability at 12 months': R_DSM_12m, 'Reliability at 18 months': R_DSM_18m, 'Reliability at 24 months': R_DSM_24m}
    simulation_results.append(result)
    R_simulation.append(R_DSM)
    graph_labels.append('EPS_redundancy: ' + str(EPS_redundancy[iteration]) + ', initial: ' + str(DSM_initial_amount[iteration]) + ', relaunch: ' + str(DSM_relaunch_amount[iteration]) + ', rate: ' + str(relaunch_rate[iteration]))
df = pd.DataFrame(simulation_results)
    


## Outputs
The output of this program provides both a comparison table of all case scenarios, the reliability of their systems at certain months in time and a graphic that compares the curve of each of them.

In [ ]:
markdown_table = df.to_markdown()
Markdown(markdown_table)

In [ ]:
plt.figure(1)

for cont in range(len(R_simulation)):
    plt.plot(Time_months / 12, R_simulation[cont], label=graph_labels[cont])

plt.xlabel('Time (years)')
plt.ylabel('Reliability of the DSM (-)')
plt.legend()
plt.grid(True)
plt.show()

## References
Guo, J., Monas, L., & Gill, E. (2014). Statistical analysis and modelling of small satellite reliability. Acta Astronautica, 98, 97–110. https://doi.org/10.1016/j.actaastro.2014.01.018

Bouwmeester, J., Menicucci, A., & Gill, E. K. A. (2022). Improving CubeSat reliability: Subsystem redundancy or improved testing? Reliability Engineering & System Safety, 220, 108288. https://doi.org/10.1016/J.RESS.2021.108288

Menchinelli, A., Ingiosi, F., Pamphili, L., Marzioli, P., Patriarca, R., Costantino, F., & Piergentili, F. (2018). A reliability engineering approach for managing risks in CubeSats. Aerospace, 5(4). https://doi.org/10.3390/aerospace5040121

Kiselyov, I. (2020). Model-Based Reliability Engineering (Delft University of Technology). Retrieved from http://resolver.tudelft.nl/uuid:3083351a-dcd1-4bb7-8169-d13055a95388

Qaisar, S. U., Ryan, M. J., & Tuttle, S. L. (2016). A Framework for Small Satellite Architecture Design. INCOSE International Symposium, 26(1), 1747–1758. https://doi.org/10.1002/j.2334-5837.2016.00258.x

Nieto-Peroy, C., & Emami, M. R. (2019). CubeSat mission: From design to operation. Applied Sciences (Switzerland), 9(15), 1–24. https://doi.org/10.3390/app9153110

Villela, T., Costa, C. A., Brandão, A. M., Bueno, F. T., & Leonardi, R. (2019). Towards the Thousandth CubeSat: A Statistical Overview. International Journal of Aerospace Engineering, 2019, 1–13. https://doi.org/10.1155/2019/5063145